In [20]:
import pandas as pd
from bs4 import BeautifulSoup
import requests
import re
import os
import time
from dotenv import load_dotenv
from google import genai

HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                         "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120 Safari/537.36"}

In [21]:
load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")
client = genai.Client(api_key=api_key)

In [31]:
BATCH_SIZE = 15                 # orgs per Gemini request (fewer requests = safer on limits)

In [23]:
def crawl_site(url):
    if not url.startswith("http"):
        return ""
 
    page = requests.get(url, headers=HEADERS, timeout=15)
    soup = BeautifulSoup(page.text, "html.parser")
 
    links = soup.find_all("a")
    good_links = []
    for link in links:
        href = link.get("href")
        if href and url in href:
            good_links.append(href)
    good_links = list(set(good_links))[:6]
 
    all_text = soup.get_text(" ", strip=True)
    for link in good_links:
        try:
            p = requests.get(link, headers=HEADERS, timeout=15)
            s = BeautifulSoup(p.text, "html.parser")
            all_text += " " + s.get_text(" ", strip=True)
        except Exception:
            continue
 
    return all_text

In [25]:
def ask_gemini_batch(names, texts):
    # build one prompt containing several orgs, each clearly numbered
    prompt = "For each organization below, list the services it offers based on its website text.\n"
    prompt += "Reply with one line per organization in this exact format:\n"
    prompt += "NUMBER. services separated by commas\n\n"
 
    for n in range(len(names)):
        prompt += f"--- ORGANIZATION {n+1}: {names[n]} ---\n"
        prompt += texts[n][:2500] + "\n\n"
 
    response = client.models.generate_content(
        model="gemini-flash-latest",
        contents=prompt
    )
    return response.text

In [ ]:
df = pd.read_csv("../../GWIorgs_v4.csv")
df = df.fillna("")

df = df[10:].reset_index(drop=True) 

# 1) crawl every org first (no Gemini calls here)
print("Crawling all sites...")
names = []
texts = []
for i in range(len(df)):
    name = df.loc[i, "Name"]
    url = df.loc[i, "URL"]
    print(f"  [{i+1}/{len(df)}] {name}")
    try:
        text = crawl_site(url)
    except Exception:
        text = ""
    names.append(name)
    texts.append(text)

# 2) send to Gemini in batches
print("\nAsking Gemini in batches...")
results = []
for start in range(0, len(names), BATCH_SIZE):
    batch_names = names[start:start + BATCH_SIZE]
    batch_texts = texts[start:start + BATCH_SIZE]

    print(f"  batch starting at org {start+1}")
    try:
        answer = ask_gemini_batch(batch_names, batch_texts)
    except Exception as e:
        answer = ""
        print(f"    batch failed: {e}")

    # split Gemini's answer back into one line per org
    lines = answer.split("\n")
    for n in range(len(batch_names)):
        found = ""
        for line in lines:
            if line.strip().startswith(f"{n+1}."):
                found = line.split(".", 1)[1].strip()
                break
        if found == "":
            found = "(no answer)"
        results.append({"Name": batch_names[n], "Services_Found": found})

    time.sleep(5)

# 3) save
report = pd.DataFrame(results)
report.to_csv("gemini_services_results_2.csv", index=False)
print()
print("done. Saved to gemini_services_results.csv")

Crawling all sites...
  [1/55] Elliot Community Services
  [2/55] Esperanza Academy
  [3/55] Essex County Habitat For Humanity
  [4/55] Family Services of Merrimack Valley/LMCC
  [5/55] Greater Lawrence Community Action Council (GLCAC)
  [6/55] Greater Lawrence Community Boating
  [7/55] Greater Lawrence Family Health
  [8/55] Greater Lawrence Fellowship of the Arts
  [9/55] Greater Lawrence Technical School
  [10/55] Groundwork Lawrence
  [11/55] Hands to Help
  [12/55] International Institute of Greater Lawrence
  [13/55] Jeanne Geiger Crisis Center
  [14/55] Lawrence Boys and Girls Club
  [15/55] Lawrence Catholic Academy
  [16/55] Lawrence Community Works
  [17/55] Lawrence General Hospital
  [18/55] Lawrence Partnership
  [19/55] Lawrence Partnership for Transition to Employment (LPTE)
  [20/55] Lawrence Prospera- Quintana Center
  [21/55] Lawrence Prospera- SISU
  [22/55] Lawrence Public Library
  [23/55] Lawrence Public Schools
  [24/55] Lawrence YMCA
  [25/55] Lazarus House Min

In [15]:
report = pd.DataFrame(results)
report.to_csv("gemini_services_results.csv", index=False)
 
print("\nDone! Saved to gemini_services_results.csv")


Done! Saved to gemini_services_results.csv


In [30]:
print(df.loc[10, "Name"])

Elliot Community Services
